# V05 - Relacoes e decisoes de modelagem

**Objetivo:** modelar uma relacao direta entre nodes, demonstrar um `edge` e discutir navegacao reversa.

**Conceitos:**
- **Relacao direta (direct relation):** um node aponta para outro por uma propriedade (ex.: `parent`).
- **Edge:** uma aresta independente que conecta dois nodes (start -> end), util para relacoes muitos-para-muitos.
- **Navegacao reversa:** definida na `View`, nao gravada como segunda copia da relacao.

In [ ]:
# Carrega a configuracao local e cria o cliente CDF. / Load local configuration and create the CDF client.
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from cognite.client import ClientConfig, CogniteClient
from cognite.client.credentials import OAuthClientCredentials, Token

env_file = Path('.env')
pkg_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env.example').exists()),
    Path.cwd(),
)

if env_file.exists():
    load_dotenv(env_file)
else:
    for name in ('.env', '.env.example'):
        candidate = pkg_root / name
        if candidate.exists():
            load_dotenv(candidate)
            break

if not os.environ.get('CDF_CLUSTER'):
    os.environ['CDF_CLUSTER'] = input('CDF_CLUSTER: ').strip()
if not os.environ.get('CDF_PROJECT'):
    os.environ['CDF_PROJECT'] = input('CDF_PROJECT: ').strip()

base_url = os.environ.get('CDF_URL', '').rstrip('/') or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
oauth_ready = env_file.exists() and all(
    os.environ.get(key) for key in ('IDP_TOKEN_URL', 'IDP_CLIENT_ID', 'IDP_CLIENT_SECRET')
)
if oauth_ready:
    scopes = os.environ.get('IDP_SCOPES', '').replace(',', ' ').split() or [f'{base_url}/.default']
    audience = os.environ.get('IDP_AUDIENCE', '')
    credentials = OAuthClientCredentials(
        token_url=os.environ['IDP_TOKEN_URL'],
        client_id=os.environ['IDP_CLIENT_ID'],
        client_secret=os.environ['IDP_CLIENT_SECRET'],
        scopes=scopes,
        **({'audience': audience} if audience else {}),
    )
else:
    # Sem .env nesta pasta: usa Bearer Token.
    bearer_token = (os.environ.get('CDF_BEARER_TOKEN') or getpass('CDF_BEARER_TOKEN: ')).strip()
    if bearer_token.lower().startswith('bearer '):
        bearer_token = bearer_token[7:].strip()
    credentials = Token(bearer_token)

client = CogniteClient(ClientConfig(
    client_name='radix-cdf-onboarding-v05',
    base_url=base_url,
    project=os.environ['CDF_PROJECT'],
    credentials=credentials,
))

## 1. Criar contrato com relacao direta
O container tem `name` (texto) e `parent` (relacao direta).

In [ ]:
from uuid import uuid4
from cognite.client.data_classes.data_modeling import (
    ContainerApply, ContainerId, DirectRelationReference, EdgeApply, EdgeId,
    MappedPropertyApply, NodeApply, NodeId, NodeOrEdgeData, SpaceApply, Text,
    ViewApply, ViewId,
)

run = uuid4().hex[:8]
sp = f'sp_ur_training_v05_{run}'
container_id = ContainerId(sp, 'Equipment')
view_id = ViewId(sp, 'Equipment', 'v1')

client.data_modeling.spaces.apply(SpaceApply(space=sp, name='UR training - V05'))
client.data_modeling.containers.apply(ContainerApply._load({
    'space': sp,
    'externalId': container_id.external_id,
    'properties': {
        'name': {'type': Text().dump(), 'nullable': False},
        'parent': {'type': {'type': 'direct'}, 'nullable': True},
    },
    'usedFor': 'node',
}))
client.data_modeling.views.apply(ViewApply(
    space=sp,
    external_id=view_id.external_id,
    version=view_id.version,
    properties={
        'name': MappedPropertyApply(container=container_id, container_property_identifier='name'),
        'parent': MappedPropertyApply(container=container_id, container_property_identifier='parent'),
    },
))
print('contrato criado em', sp)

## 2. Criar nodes com relacao direta
`child.parent` aponta para `parent` via `DirectRelationReference`.

In [ ]:
parent_id = NodeId(sp, 'eq-parent')
child_id = NodeId(sp, 'eq-child')
client.data_modeling.instances.apply(nodes=[
    NodeApply(space=sp, external_id=parent_id.external_id,
              sources=[NodeOrEdgeData(source=view_id, properties={'name': 'Linha de producao'})]),
    NodeApply(space=sp, external_id=child_id.external_id,
              sources=[NodeOrEdgeData(source=view_id, properties={
                  'name': 'Bomba 01',
                  'parent': {'space': sp, 'externalId': parent_id.external_id},
              })]),
])
child = client.data_modeling.instances.retrieve_nodes(nodes=child_id, sources=view_id)
print('relacao direta de child.parent:', child.properties[view_id].get('parent'))

## 3. Criar um edge
Um edge conecta start -> end de forma independente da propriedade direta.

In [ ]:
edge_id = EdgeId(sp, 'eq-edge-connects')
client.data_modeling.instances.apply(edges=EdgeApply(
    space=sp,
    external_id=edge_id.external_id,
    type=DirectRelationReference(sp, 'connects'),
    start_node=DirectRelationReference(sp, parent_id.external_id),
    end_node=DirectRelationReference(sp, child_id.external_id),
))
edges = client.data_modeling.instances.list(instance_type='edge', space=sp, limit=10)
edges.to_pandas()

## Direta vs edge: quando usar
- **Relacao direta:** cardinalidade 1:1 ou 1:N estavel, navegacao previsivel, menor custo.
- **Edge:** N:N, relacoes com atributos proprios, ou quando o vinculo evolui de forma independente dos nodes.
- **Reversa:** declarada na View (connection) para navegar de `parent` para seus filhos sem duplicar dados.

## Mini-exercicio
- Crie um segundo filho e liste os dois edges saindo do `parent`.
- Esboce (em markdown) como declararia a navegacao reversa `children` na View.

## 4. Limpeza idempotente
Exclui edge -> nodes -> view -> container -> space.

> Detalhe importante: o `type` do edge (`connects`) materializa um node proprio no space. Ele tambem precisa ser removido, senao a exclusao do space falha com "space contains nodes or edges". A exclusao do space usa retry porque a remocao e eventualmente consistente.

In [ ]:
import time
from cognite.client.exceptions import CogniteAPIError

assert sp.startswith('sp_ur_training_v05_')
client.data_modeling.instances.delete(edges=edge_id)
client.data_modeling.instances.delete(nodes=[parent_id, child_id, NodeId(sp, 'connects')])
client.data_modeling.views.delete(view_id)
client.data_modeling.containers.delete(container_id)
for attempt in range(10):
    try:
        client.data_modeling.spaces.delete(sp)
        break
    except CogniteAPIError as exc:
        if 'contain nodes or edges' in str(exc) and attempt < 9:
            time.sleep(2)
            continue
        raise
print('space_still_exists:', client.data_modeling.spaces.retrieve(sp) is not None)